# Libraries and data load

In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.tree import export_text
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv')

In [3]:
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


# Data Preparation

Check null count and dtype

In [4]:
null_counts_per_column = df.isnull().sum()

for col in df.columns:
    if null_counts_per_column[col] > 0:
        print(f"{col} ({df[col].dtype}): {null_counts_per_column[col]} nulls")

num_cylinders (float64): 482 nulls
horsepower (float64): 708 nulls
acceleration (float64): 930 nulls
num_doors (float64): 502 nulls


Fill nulls

Split dataset

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [6]:
dv = DictVectorizer(sparse=False)

In [7]:
def prep_train_df(df, target):
    val_dicts = df.fillna(0).drop(columns=[target]).to_dict(orient='records')
    return dv.fit_transform(val_dicts)

In [8]:
target = 'fuel_efficiency_mpg'

X_train = prep_train_df(df_train, 'fuel_efficiency_mpg')
X_test = prep_train_df(df_test, 'fuel_efficiency_mpg')
X_val = prep_train_df(df_val, 'fuel_efficiency_mpg')

y_train = df_train[target].values
y_test = df_test[target].values
y_val = df_val[target].values

# Questions

## Q1 Decision Tree

In [9]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [10]:
root_feature_index = dt.tree_.feature[0]
vec_name = dv.get_feature_names_out()[root_feature_index]
print(vec_name)

vehicle_weight


## Q2: Random Forest

In [11]:
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,10
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("Validation RMSE:", rmse)

Validation RMSE: 0.4599777557336148


## Q3: RF Estimators

In [13]:
for estimator in list(range(10, 210, 10)):
    rf = RandomForestRegressor(n_estimators=estimator, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = round(np.sqrt(mean_squared_error(y_val, y_pred)), 3)

    print(f"Estimator = {estimator} RMSE:", rmse)

Estimator = 10 RMSE: 0.46
Estimator = 20 RMSE: 0.454
Estimator = 30 RMSE: 0.451
Estimator = 40 RMSE: 0.448
Estimator = 50 RMSE: 0.446
Estimator = 60 RMSE: 0.445
Estimator = 70 RMSE: 0.445
Estimator = 80 RMSE: 0.445
Estimator = 90 RMSE: 0.445
Estimator = 100 RMSE: 0.444
Estimator = 110 RMSE: 0.443
Estimator = 120 RMSE: 0.444
Estimator = 130 RMSE: 0.443
Estimator = 140 RMSE: 0.443
Estimator = 150 RMSE: 0.443
Estimator = 160 RMSE: 0.443
Estimator = 170 RMSE: 0.443
Estimator = 180 RMSE: 0.442
Estimator = 190 RMSE: 0.443
Estimator = 200 RMSE: 0.443


## Q4: RF Depth

In [14]:
results = []

for depth in [10, 15, 20, 25]:
    for estimator in range(10, 210, 10):
        rf = RandomForestRegressor(
            n_estimators=estimator,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        results.append({
            'max_depth': depth,
            'n_estimators': estimator,
            'rmse': rmse
        })


In [15]:
df_results = pd.DataFrame(results)

summary = df_results.groupby('max_depth')['rmse'].mean().reset_index()
print(summary)

best_depth = summary.loc[summary['rmse'].idxmin(), 'max_depth']
print("\nBest max_depth based on mean RMSE:", best_depth)

   max_depth      rmse
0         10  0.442321
1         15  0.445060
2         20  0.445644
3         25  0.445661

Best max_depth based on mean RMSE: 10


## Q5: Feature importance

In [16]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

,n_estimators,10
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
importances = rf.feature_importances_
feature_names = dv.get_feature_names_out()

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(10))

                feature  importance
13       vehicle_weight    0.959162
6            horsepower    0.016040
0          acceleration    0.011471
3   engine_displacement    0.003269
7            model_year    0.003182
8         num_cylinders    0.002359
9             num_doors    0.001591
12           origin=USA    0.000555
11        origin=Europe    0.000520
10          origin=Asia    0.000476


## Q6: XGBoost eta

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )
    
    y_pred = model.predict(dval)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"eta = {eta}, validation RMSE = {rmse:.3f}")

eta = 0.3, validation RMSE = 0.450
eta = 0.1, validation RMSE = 0.426


---- End ----